In [1]:
module load gatk
module load picard
module load datasets
module load igenomes/hg38

module load datasets
What it does:
Loads a module called datasets into your environment.
Why?
This is likely a local or HPC-specific module providing access to curated or example data sets used for testing, training, or benchmarking pipelines. It might include sample FASTQ, BAM, VCF, or annotation files.
Note: This module’s exact content depends on the HPC environment. On some systems, it might contain public reference data or example project data.

module load igenomes/hg38
What it does:
Loads the igenomes/hg38 module into your environment.
Why?
iGenomes is a collection of reference genomes and annotations pre-packaged for popular aligners and tools (e.g., BWA, STAR, GATK, Picard).
In this case:
hg38 refers to the GRCh38/hg38 human reference genome
This module would give your pipeline access to the GRCh38 reference FASTA file, index files, annotation GTF files, known variant databases, and other required resources, all organized in a standardized directory structure.
iGenomes is valuable because it saves you the trouble of downloading, formatting, and indexing reference files manually — it ensures compatibility and reproducibility across tools.


✅ What This Whole Block Does:
It prepares your HPC environment by loading essential tools (GATK, Picard), reference data (GRCh38/hg38 genome), and possibly example datasets for a variant calling or NGS analysis workflow. This is a typical setup block you’d run before executing a bioinformatics pipeline that involves sequence alignment, post-processing, and variant calling.

In [ ]:
#Enter the directory where you want to work
cd ~/bigcare/Leslie

In [70]:
my_list=("COV362_sort.bam.RG.bam" \
         "Hec50_sort.bam.RG.bam" \
         "Ishikawa_sort.bam.RG.bam" \
         "KLE_sort.bam.RG.bam" \
         "PDX1_sort.bam.RG.bam" \
         "PDX2_sort.bam.RG.bam" \
         "PDX3_sort.bam.RG.bam" \
         "PEO1_sort.bam.RG.bam")

for file in "${my_list[@]}"; do
 echo "file is $file"
 end="alignment_metrics.txt"
 out="${file}_${end}"
 echo "out is $out"
 
 picard CollectAlignmentSummaryMetrics \
  R=WholeGenomeFasta/genome.fa \
  I=$file \
  O=$out
 
done

file is COV362_sort.bam.RG.bam
out is COV362_sort.bam.RG.bam_alignment_metrics.txt
INFO	2024-10-24 13:15:42	CollectAlignmentSummaryMetrics	

********** NOTE: Picard's command line syntax is changing.
**********
********** For more information, please see:
********** https://github.com/broadinstitute/picard/wiki/Command-Line-Syntax-Transition-For-Users-(Pre-Transition)
**********
********** The command line looks like this in the new syntax:
**********
**********    CollectAlignmentSummaryMetrics -R WholeGenomeFasta/genome.fa -I COV362_sort.bam.RG.bam -O COV362_sort.bam.RG.bam_alignment_metrics.txt
**********


13:15:42.476 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/apps/spack/anvil/apps/picard/2.25.7-gcc-11.2.0-jjgxug4/bin/picard.jar!/com/intel/gkl/native/libgkl_compression.so
[Thu Oct 24 13:15:42 EDT 2024] CollectAlignmentSummaryMetrics INPUT=COV362_sort.bam.RG.bam OUTPUT=COV362_sort.bam.RG.bam_alignment_metrics.txt REFERENCE_SEQUENCE=WholeGenomeFasta/gen

What does R=WholeGenomeFasta/genome.fa mean?
This is the reference genome that all your reads (from BAM files) are aligned to, and it serves as the basis for identifying variants. It is not the genome of your samples or the cancer panel.

✅ So to clarify:
R=WholeGenomeFasta/genome.fa refers to:
➤ The full reference genome of the species you're working on — e.g., human genome (hg38 or hg19).

It's what your BAM files were aligned to during the alignment step.

It's required by GATK tools to compare sample reads to a standard coordinate system.

Even if your sequencing is targeted (e.g., cancer panel), you still use the whole reference genome file unless you're using a custom reference (rare).

🧪 But what about the cancer panel?
Cancer panels target only specific genes/regions (e.g., 50–500 genes).

However, the BAM files still contain the reads aligned to positions in the whole reference genome — even if most regions have no reads.

Your variant caller will only report variants in the regions covered by your reads — or if you supply a BED file using -L to restrict calling.

🧬 Why not use a "panel genome"?
Because:

Most bioinformatics tools expect a complete, indexed reference genome (FASTA + .fai + .dict).

The tools don’t work properly with “partial” or custom panel references unless they’re very carefully formatted.

🔬 Summary
Component	What it represents
R=WholeGenomeFasta/genome.fa	:The full reference genome (e.g., hg38) used for alignment and variant calling
BAM files	    :Your sample’s aligned sequencing reads (from a cancer panel)
Cancer Panel	:Targets a subset of genes; doesn’t affect which reference genome you use



What it does:
Defines an array variable in bash called my_list.
Each item in the array is the name of a BAM file you want to process.
The \ at the end of each line is a line continuation character, telling bash to treat this as a single continuous command.

echo "file is $file"
What it does:
Prints the name of the current file being processed in this iteration.
Helpful for tracking progress or debugging.

end="alignment_metrics.txt"
What it does:
Defines a string variable called end with the value alignment_metrics.txt.
This will be used to name the output file later.

out="${file}_${end}"
What it does:
Creates a new variable out by concatenating the current file name with _alignment_metrics.txt.This will be the output file for the metrics generated by Picard for that BAM file.
Example:
If file is COV362_sort.bam.RG.bam, then out will be COV362_sort.bam.RG.bam_alignment_metrics.txt
Example: If file is COV362_sort.bam.RG.bam, then out will be COV362_sort.bam.RG.bam_alignment_metrics.txt

echo "out is $out"
What it does:
Prints the name of the output file for the current BAM file.
Another helpful progress/debugging message.


 picard CollectAlignmentSummaryMetrics \
  R=WholeGenomeFasta/genome.fa \
  I=$file \
  O=$out
What it does:

Runs the Picard CollectAlignmentSummaryMetrics tool.
R=WholeGenomeFasta/genome.fa → Specifies the reference genome FASTA file (required for alignment summary metrics).
I=$file → Specifies the input BAM file (the current file from the loop).
O=$out → Specifies the output metrics file name, generated earlier.
CollectAlignmentSummaryMetrics collects and outputs metrics about the alignment of reads within the BAM file against the reference genome (e.g., total reads, mapped reads, insert size, base quality distribution, etc.).

done
Closes the for loop.
Once all BAM files in my_list are processed, the script stops.

In [74]:
my_list=("COV362_sort.bam.RG.bam" \
         "Hec50_sort.bam.RG.bam" \
         "Ishikawa_sort.bam.RG.bam" \
         "KLE_sort.bam.RG.bam" \
         "PDX1_sort.bam.RG.bam" \
         "PDX2_sort.bam.RG.bam" \
         "PDX3_sort.bam.RG.bam" \
         "PEO1_sort.bam.RG.bam")

for file in "${my_list[@]}"; do
 echo "file is $file"
 end="insert_metrics.txt"
 histend="insert_hist.pdf"
 out="${file}_${end}"
 hist="${file}_${histend}"
 echo "out is $out"
 echo "hist is $hist"
 
picard CollectInsertSizeMetrics \
        INPUT=$file \
        OUTPUT=$out \
        HISTOGRAM_FILE=$hist 
done

file is COV362_sort.bam.RG.bam
out is COV362_sort.bam.RG.bam_insert_metrics.txt
hist is COV362_sort.bam.RG.bam_insert_hist.pdf
INFO	2024-10-24 13:40:33	CollectInsertSizeMetrics	

********** NOTE: Picard's command line syntax is changing.
**********
********** For more information, please see:
********** https://github.com/broadinstitute/picard/wiki/Command-Line-Syntax-Transition-For-Users-(Pre-Transition)
**********
********** The command line looks like this in the new syntax:
**********
**********    CollectInsertSizeMetrics -INPUT COV362_sort.bam.RG.bam -OUTPUT COV362_sort.bam.RG.bam_insert_metrics.txt -HISTOGRAM_FILE COV362_sort.bam.RG.bam_insert_hist.pdf
**********


INFO	2024-10-24 13:40:33	RExecutor	Executing R script via command: Rscript /tmp/script7583390227491460108.R
INFO	2024-10-24 13:40:33	ProcessExecutor	[1] "Checking if R is installed"
13:40:33.373 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/apps/spack/anvil/apps/picard/2.25.7-gcc-11.2.0-jjgx

histend="insert_hist.pdf"
Defines another suffix
This will be used to name the histogram output file (in PDF format)


picard CollectInsertSizeMetrics \
        INPUT=$file \
        OUTPUT=$out \
        HISTOGRAM_FILE=$hist 

Runs Picard’s CollectInsertSizeMetrics tool
This command performs the following:
INPUT=$file → Tells Picard which BAM file to analyze.
OUTPUT=$out → Where to save the insert size metrics (text report).
HISTOGRAM_FILE=$hist → Where to save the graphical histogram (PDF).
📌 This tool reports insert size distribution, i.e., the distance between paired-end reads (distance between the start of the first read (Read 1 or R1) and the start of the second read (Read 2 or R2) on the original DNA fragment.= insert size)— which helps assess library prep quality, duplication artifacts, and expected fragment length.

done
Ends the for loop
The loop will repeat for each BAM file in my_list.


What the Full Script Does:
For each BAM file in the list, the script:
Constructs two output filenames.
Runs Picard's CollectInsertSizeMetrics to:
Generate insert size metrics (.txt)
Generate a histogram (.pdf)
Prints progress info for traceability.

In [ ]:
# BQSR next: Base Quality Score Recalibration,

my_list=("COV362_sort.bam.RG.bam" \
         "Hec50_sort.bam.RG.bam" \
         "Ishikawa_sort.bam.RG.bam" \
         "KLE_sort.bam.RG.bam" \
         "PDX1_sort.bam.RG.bam" \
         "PDX2_sort.bam.RG.bam" \
         "PDX3_sort.bam.RG.bam" \
         "PEO1_sort.bam.RG.bam")

for file in "${my_list[@]}"; do
 echo "file is $file"
 end="recal_data.table"
 out="${file}_${end}"
 echo "out is $out"
 
 gatk BaseRecalibrator \
  -R WholeGenomeFasta/genome.fa \
  -I $file \
  --output $out \
  --known-sites /anvil/datasets/igenomes/Homo_sapiens/GATK/hg38/1000G_phase1.snps.high_confidence.hg38.vcf.gz \
  --known-sites /anvil/datasets/igenomes/Homo_sapiens/GATK/hg38/Mills_and_1000G_gold_standard.indels.hg38.vcf.gz 

done


file is COV362_sort.bam.RG.bam
out is COV362_sort.bam.RG.bam_recal_data.table
Using GATK jar /apps/spack/anvil/apps/gatk/4.1.8.1-gcc-11.2.0-ash2pfh/bin/gatk-package-4.1.8.1-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /apps/spack/anvil/apps/gatk/4.1.8.1-gcc-11.2.0-ash2pfh/bin/gatk-package-4.1.8.1-local.jar BaseRecalibrator -R WholeGenomeFasta/genome.fa -I COV362_sort.bam.RG.bam --output COV362_sort.bam.RG.bam_recal_data.table --known-sites /anvil/datasets/igenomes/Homo_sapiens/GATK/hg38/1000G_phase1.snps.high_confidence.hg38.vcf.gz --known-sites /anvil/datasets/igenomes/Homo_sapiens/GATK/hg38/Mills_and_1000G_gold_standard.indels.hg38.vcf.gz
14:07:53.228 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/apps/spack/anvil/apps/gatk/4.1.8.1-gcc-11.2.0-ash2pfh/bin/gatk-package-4.1.8.1-local.jar!/com/intel/gkl/native/libgkl_comp

my_list=("COV362_sort.bam.RG.bam" \
         "Hec50_sort.bam.RG.bam" \
         "Ishikawa_sort.bam.RG.bam" \
         "KLE_sort.bam.RG.bam" \
         "PDX1_sort.bam.RG.bam" \
         "PDX2_sort.bam.RG.bam" \
         "PDX3_sort.bam.RG.bam" \
         "PEO1_sort.bam.RG.bam")
Defines an array of BAM file names that are sorted and have read group (RG) tags added. These are the input files that will go through BQSR.

end="recal_data.table"
out="${file}_${end}"
Sets a variable end to the suffix used for the BQSR output table AND Constructs the full output file name by appending _recal_data.table to the original file name.

If file="COV362_sort.bam.RG.bam", then
out="COV362_sort.bam.RG.bam_recal_data.table"


gatk BaseRecalibrator \
  -R WholeGenomeFasta/genome.fa \
  -I $file \
  --output $out \
  --known-sites /anvil/datasets/igenomes/Homo_sapiens/GATK/hg38/1000G_phase1.snps.high_confidence.hg38.vcf.gz \
  --known-sites /anvil/datasets/igenomes/Homo_sapiens/GATK/hg38/Mills_and_1000G_gold_standard.indels.hg38.vcf.gz
Runs GATK’s BaseRecalibrator tool, which analyzes patterns of mismatches and adjusts base quality scores accordingly.

-R WholeGenomeFasta/genome.fa → Specifies the reference genome FASTA file (must match the alignment).
-I $file → The input BAM file for which base qualities will be recalibrated.
--output $out → Output file to store recalibration data (.table format).
--known-sites ...vcf.gz → Lists of known variant sites (SNPs and indels) used to mask real variants so they aren’t misinterpreted as sequencing errors.
1000G SNPs: common SNP variants
Mills & 1000G: validated indels
These known variants are critical for distinguishing true variation from machine artifacts.


What This Script Does in Plain English:
For each BAM file, it:
Logs the file being processed.
Creates an output name for storing recalibration data.
Runs GATK’s BaseRecalibrator to analyze and correct base quality scores using known human variant sites (GRCh38).
Repeats the process for all samples in the list.



In [7]:
my_list=("COV362_sort.bam.RG.bam" \
         "Hec50_sort.bam.RG.bam" \
         "Ishikawa_sort.bam.RG.bam" \
         "KLE_sort.bam.RG.bam" \
         "PDX1_sort.bam.RG.bam" \
         "PDX2_sort.bam.RG.bam" \
         "PDX3_sort.bam.RG.bam" \
         "PEO1_sort.bam.RG.bam")

for file in "${my_list[@]}"; do
 echo "file is $file"
 end="recal_data.table"
 bqsr="${file}_${end}"
 echo "bqsr is $bqsr"
 out="${file}_recal.bam"
 echo "out is $out"
 
 gatk ApplyBQSR \
  -R WholeGenomeFasta/genome.fa \
  -I $file \
  -bqsr $bqsr \
  -O $out
done

file is COV362_sort.bam.RG.bam
bqsr is COV362_sort.bam.RG.bam_recal_data.table
out is COV362_sort.bam.RG.bam_recal.bam
Using GATK jar /apps/spack/anvil/apps/gatk/4.1.8.1-gcc-11.2.0-ash2pfh/bin/gatk-package-4.1.8.1-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /apps/spack/anvil/apps/gatk/4.1.8.1-gcc-11.2.0-ash2pfh/bin/gatk-package-4.1.8.1-local.jar ApplyBQSR -R WholeGenomeFasta/genome.fa -I COV362_sort.bam.RG.bam -bqsr COV362_sort.bam.RG.bam_recal_data.table -O COV362_sort.bam.RG.bam_recal.bam
14:50:34.124 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/apps/spack/anvil/apps/gatk/4.1.8.1-gcc-11.2.0-ash2pfh/bin/gatk-package-4.1.8.1-local.jar!/com/intel/gkl/native/libgkl_compression.so
Oct 24, 2024 2:50:34 PM shaded.cloud_nio.com.google.auth.oauth2.ComputeEngineCredentials runningOnComputeEngine
INFO: Failed to detect whethe

 gatk ApplyBQSR \
  -R WholeGenomeFasta/genome.fa \
  -I $file \
  -bqsr $bqsr \
  -O $out

Runs GATK's ApplyBQSR tool, which applies the recalibration data to the original BAM file.
Breakdown of the command:
-R WholeGenomeFasta/genome.fa → The reference genome used during alignment and recalibration
-I $file → The original BAM file
-bqsr $bqsr → The BQSR table created by BaseRecalibrator
-O $out → The output BAM file, now with recalibrated base quality scores
Real-World Context

This is the second step in the BQSR process:
BaseRecalibrator builds the error model.
ApplyBQSR corrects the base quality scores in the BAM file using that model.



In [5]:
#2nd round - Not Needed - Skipped for now.
gatk BaseRecalibrator \
 -R WholeGenomeFasta/genome.fa \
 -I PDX2_recal_reads.bam \
 -O post_recal_data.table \
 --known-sites /anvil/datasets/igenomes/Homo_sapiens/GATK/hg38/1000G_phase1.snps.high_confidence.hg38.vcf.gz \
 --known-sites /anvil/datasets/igenomes/Homo_sapiens/GATK/hg38/Mills_and_1000G_gold_standard.indels.hg38.vcf.gz 

Using GATK jar /apps/spack/anvil/apps/gatk/4.1.8.1-gcc-11.2.0-ash2pfh/bin/gatk-package-4.1.8.1-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /apps/spack/anvil/apps/gatk/4.1.8.1-gcc-11.2.0-ash2pfh/bin/gatk-package-4.1.8.1-local.jar BaseRecalibrator -R WholeGenomeFasta/genome.fa -I PDX2_recal_reads.bam -O post_recal_data.table --known-sites /anvil/datasets/igenomes/Homo_sapiens/GATK/hg38/1000G_phase1.snps.high_confidence.hg38.vcf.gz --known-sites /anvil/datasets/igenomes/Homo_sapiens/GATK/hg38/Mills_and_1000G_gold_standard.indels.hg38.vcf.gz
11:10:59.661 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/apps/spack/anvil/apps/gatk/4.1.8.1-gcc-11.2.0-ash2pfh/bin/gatk-package-4.1.8.1-local.jar!/com/intel/gkl/native/libgkl_compression.so
Oct 24, 2024 11:10:59 AM shaded.cloud_nio.com.google.auth.oauth2.ComputeEngineCredentials run

gatk BaseRecalibrator \
gatk: This calls the Genome Analysis Toolkit (GATK), a software package for variant discovery in high-throughput sequencing data.
BaseRecalibrator: This is the GATK tool used for Base Quality Score Recalibration (BQSR). It analyzes patterns of covariation in base quality scores to model systematic errors made by the sequencer when estimating base quality.


 -R WholeGenomeFasta/genome.fa \
-R: Specifies the reference genome in FASTA format.
WholeGenomeFasta/genome.fa: Path to the reference genome file against which reads were aligned and to which known variants are referenced.

 -I PDX2_recal_reads.bam \
 -I: Input flag.
PDX2_recal_reads.bam: The input BAM file, which contains the aligned sequence reads for sample PDX2. This is the file that will be evaluated for base quality recalibration.


--known-sites /anvil/datasets/igenomes/Homo_sapiens/GATK/hg38/1000G_phase1.snps.high_confidence.hg38.vcf.gz \
--known-sites: This flag provides known variant sites (in VCF format) to exclude true variants from being mistaken as sequencing errors.
The file here (1000G_phase1.snps.high_confidence.hg38.vcf.gz) contains known SNPs from the 1000 Genomes Project.


 --known-sites /anvil/datasets/igenomes/Homo_sapiens/GATK/hg38/Mills_and_1000G_gold_standard.indels.hg38.vcf.gz
 Another --known-sites file, this time containing known INDELs (insertions and deletions).
The Mills and 1000G set is a gold-standard set of known indels used by GATK best practices.

This command generates a recalibration model for correcting systematic biases in base quality scores. It uses known variant sites to avoid penalizing true biological variation. The output is a .table file (post_recal_data.table) used to improve accuracy in downstream variant calling.



In [9]:
# - Not Needed - Skipped for now.
gatk AnalyzeCovariates -before recal_data.table -after post_recal_data.table -plots recalibration_plots.pdf

Using GATK jar /apps/spack/anvil/apps/gatk/4.1.8.1-gcc-11.2.0-ash2pfh/bin/gatk-package-4.1.8.1-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /apps/spack/anvil/apps/gatk/4.1.8.1-gcc-11.2.0-ash2pfh/bin/gatk-package-4.1.8.1-local.jar AnalyzeCovariates -before recal_data.table -after post_recal_data.table -plots recalibration_plots.pdf
11:14:48.902 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/apps/spack/anvil/apps/gatk/4.1.8.1-gcc-11.2.0-ash2pfh/bin/gatk-package-4.1.8.1-local.jar!/com/intel/gkl/native/libgkl_compression.so
Oct 24, 2024 11:14:49 AM shaded.cloud_nio.com.google.auth.oauth2.ComputeEngineCredentials runningOnComputeEngine
INFO: Failed to detect whether we are running on Google Compute Engine.
11:14:49.019 INFO  AnalyzeCovariates - ------------------------------------------------------------
11:14:49.019 INFO  A

: 3

gatk AnalyzeCovariates
This is a tool from GATK (Genome Analysis Toolkit) used in the Base Quality Score Recalibration (BQSR) workflow.
It helps visualize the effect of BQSR by comparing quality score distributions before and after recalibration.

-before recal_data.table
This is the input recalibration data table generated by the BaseRecalibrator tool before applying the recalibration.
It contains information about how the original quality scores deviate from expectations based on known variants.

-after post_recal_data.table
This is the post-recalibration data table, also generated by BaseRecalibrator, but after applying BQSR using ApplyBQSR.
It reflects how the base quality scores now align more closely with empirical error rates after recalibration.

-plots recalibration_plots.pdf
This tells the tool to generate a PDF file (recalibration_plots.pdf) that contains plots comparing the before and after quality score metrics.
These plots help verify that recalibration improved the accuracy of base quality scores.


What This Step Does (in Simple Terms)
Think of BQSR as spell-checking the quality scores that come from the sequencer.
This command visualizes the "before and after" of that spell-checking process, showing if your correction improved the reliability of the data.


Why It’s Important
Helps validate that BQSR was applied correctly.
Aids in debugging or quality control of sequencing data.
Ensures more accurate downstream variant calling.

In [42]:
gatk --java-options "-Xmx4g" HaplotypeCaller  \
   -R WholeGenomeFasta/genome.fa \
   -I PDX2_recal_reads.bam \
   -O PDX2.g.vcf.gz \
   -L chr22 \
   -ERC GVCF
# ADD -L options????
# ADD -ip option????

Using GATK jar /apps/spack/anvil/apps/gatk/4.1.8.1-gcc-11.2.0-ash2pfh/bin/gatk-package-4.1.8.1-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx4g -jar /apps/spack/anvil/apps/gatk/4.1.8.1-gcc-11.2.0-ash2pfh/bin/gatk-package-4.1.8.1-local.jar HaplotypeCaller -R WholeGenomeFasta/genome.fa -I PDX2_recal_reads.bam -O PDX2.g.vcf.gz -L chr22 -ERC GVCF
12:03:21.561 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/apps/spack/anvil/apps/gatk/4.1.8.1-gcc-11.2.0-ash2pfh/bin/gatk-package-4.1.8.1-local.jar!/com/intel/gkl/native/libgkl_compression.so
Oct 24, 2024 12:03:21 PM shaded.cloud_nio.com.google.auth.oauth2.ComputeEngineCredentials runningOnComputeEngine
INFO: Failed to detect whether we are running on Google Compute Engine.
12:03:21.685 INFO  HaplotypeCaller - ------------------------------------------------------------
12:03:21.685

gatk --java-options "-Xmx4g"
Runs the GATK toolkit and passes Java options.
-Xmx4g: Allocates 4 GB of memory to the Java Virtual Machine running GATK.

HaplotypeCaller
Specifies the GATK tool to use: HaplotypeCaller, which performs variant calling.
It analyzes the reads and reconstructs possible haplotypes to identify variants like SNPs and indels.

-R WholeGenomeFasta/genome.fa

#-R: Reference genome file.
This tells HaplotypeCaller to use the FASTA file at WholeGenomeFasta/genome.fa as the reference sequence for aligning reads and identifying variants.


-I PDX2_recal_reads.bam
-I: Input file, a BAM file of aligned sequencing reads.
This is your recalibrated BAM file for the sample PDX2 (after BQSR), which will be used for variant calling.


-O PDX2.g.vcf.gz

#-O: Output file.
This specifies that the variant calls should be written in GVCF format, compressed as a .gz file.
The output will be called PDX2.g.vcf.gz.

-L chr22
Limits variant calling to chromosome 22 only.
This is useful when testing or focusing analysis on a specific region.

-ERC GVCF

#-ERC: Emission mode, set to GVCF (Genomic VCF).
GVCF mode records both variant and non-variant sites, which is essential for joint genotyping across multiple samples later.
It's the preferred mode for pipelines that include multiple samples.


This command tells GATK to:
"Use 4 GB of memory to run HaplotypeCaller on the recalibrated BAM file for sample PDX2, using the human reference genome. Look only at chromosome 22, and output a GVCF file that contains detailed information about all positions, even those without mutations."

